# Build and Prepare Dataset
---
This notebook condenses the raw simulation output, computes the flat-sky power spectra, and assembles training-ready arrays that every other notebook in this directory reads.

More information about the package can be found here: [Documentation](https://reionemu.org/)

## Imports

In [1]:
import sys
import json
from dataclasses import asdict, is_dataclass

import h5py
import numpy as np

from reionemu import (
    CondenseConfig,
    condense_sim_root,
    ClConfig,
    add_cl_to_condensed_h5,
    BuildXYConfig,
    build_and_write_training,
    dataset_summary,
    file_fingerprint,
)

## Paths, Configs, and Constants

In [2]:
from config import RAW_PATH, H5_PATH, BUILD_INFO_PATH, PARAM_BOUNDS, param_order, SEED, make_dirs

make_dirs()

## Seeding

In [3]:
np.random.seed(SEED)

print(f"Python:\t\t{sys.version.split()[0]}")
print("-" * 30)
print(f"NumPy:\t\t{np.__version__}")
print(f"h5py:\t\t{h5py.version.version}")
print("-" * 30)
print(f"Global Seed:\t{SEED}")

Python:		3.13.1
------------------------------
NumPy:		2.5.2
h5py:		3.16.0
------------------------------
Global Seed:	42


## Rebuild Guard

`REBUILD=False` defines the configs but skips the three expensive writes, so the notebook can be run top-to-bottom to inspect the existing dataset without touching it.

In [4]:
REBUILD = True

print(f"REBUILD = {REBUILD}")
print(f"Raw Simulations:\n{RAW_PATH}")
print(f"Dataset:\n{H5_PATH}")

if H5_PATH.exists():
    fp = file_fingerprint(H5_PATH)
    print(f"\nExisting dataset: {fp['file_size_bytes'] / 1e9:.2f} GB, Modified {fp['modified_at']}")
    if REBUILD:
        print("REBUILD=True: the file above WILL BE OVERWRITTEN.")
else:
    print("\nNo dataset at H5_PATH.")
    if not REBUILD:
        raise FileNotFoundError(f"No dataset at {H5_PATH} and REBUILD=False.")

REBUILD = True
Raw Simulations:
/Users/robertxpearce/Desktop/Research/LEADS Lab/reionemu/reionemu-pasa-2026/datasets/raw/sims_v6
Dataset:
/Users/robertxpearce/Desktop/Research/LEADS Lab/reionemu/reionemu-pasa-2026/datasets/processed/publication_condensed_v6.h5

Existing dataset: 9.04 GB, Modified 2026-08-27T00:38:17.877086+00:00
REBUILD=True: the file above WILL BE OVERWRITTEN.


## Build Condensed Dataset

In [5]:
%%time

cdcfg = CondenseConfig(
    overwrite=True,
    require_obs_and_pk=True,
)

if REBUILD:
    condense_stats = condense_sim_root(
        sim_root=RAW_PATH,
        out_path=H5_PATH,
        config=cdcfg,
        file_description="Condensed simulation dataset for publication figures.",
        version=1,
    )
    print(condense_stats)
else:
    condense_stats = None
    print("REBUILD=False: Skipped, existing dataset untouched.")

CondenseStats(written=1000, skipped_missing_obs_pk=0, skipped_read_error=0, skipped_validation_error=0)
CPU times: user 1.1 s, sys: 6.18 s, total: 7.29 s
Wall time: 12.1 s


## Compute Angular Power Spectrum

In [6]:
%%time

clcfg = ClConfig(
    nbins=5,
    ell_cut=1000.0,
    overwrite=True,
)

if REBUILD:
    cl_stats = add_cl_to_condensed_h5(
        h5_path=H5_PATH,
        config=clcfg,
    )
    print(cl_stats)
else:
    cl_stats = None
    print("REBUILD=False: Skipped, existing dataset untouched.")

1000
CPU times: user 2min 50s, sys: 1.21 s, total: 2min 51s
Wall time: 2min 53s


## Build Training Arrays

In [7]:
%%time

xycfg = BuildXYConfig(
    param_names=tuple(param_order),
    y_source="dl_ksz",
    y_transform="ln",
    eps=1e-30,
)

if REBUILD:
    xy_stats = build_and_write_training(
        h5_path=H5_PATH,
        config=xycfg,
        overwrite=True,
    )
    print(xy_stats)
else:
    xy_stats = None
    print("REBUILD=False: Skipped, existing dataset untouched.")

1000
CPU times: user 258 ms, sys: 29.3 ms, total: 287 ms
Wall time: 405 ms


## Verify Dataset Contract

In [8]:
with h5py.File(H5_PATH, "r") as f:
    X_shape = f["training"]["X"].shape
    Y_shape = f["training"]["Y"].shape
    ell = f["training"]["ell"][...]

print(f"X:\t{X_shape}")
print(f"Y:\t{Y_shape}")
print()
print(f"ell Shape: {ell.shape}")
print(f"ell Bins: {np.asarray(ell)}")

assert X_shape[0] == Y_shape[0], "X and Y disagree on the number of simulations"
assert X_shape[1] == len(param_order) == PARAM_BOUNDS.shape[0], f"X has {X_shape[1]} parameters but config declares {len(param_order)} names and {PARAM_BOUNDS.shape[0]} bounds"
assert Y_shape[1] == len(ell) == clcfg.nbins, f"Y has {Y_shape[1]} bins but ell has {len(ell)} and clcfg.nbins={clcfg.nbins}"

print()
print(json.dumps(dataset_summary(H5_PATH), indent=2, default=str))

X:	(1000, 4)
Y:	(1000, 5)

ell Shape: (5,)
ell Bins: [ 2033.49633959  4093.41528716  6153.33423472  8213.25318229
 10273.17212985]

{
  "path": "/Users/robertxpearce/Desktop/Research/LEADS Lab/reionemu/reionemu-pasa-2026/datasets/processed/publication_condensed_v6.h5",
  "fingerprint": {
    "path": "/Users/robertxpearce/Desktop/Research/LEADS Lab/reionemu/reionemu-pasa-2026/datasets/processed/publication_condensed_v6.h5",
    "file_size_bytes": 9036718504,
    "modified_at": "2026-08-27T00:48:25.194188+00:00"
  },
  "n_samples": 1000,
  "n_parameters": 4,
  "n_targets": 5,
  "param_names": [
    "zmean_zre",
    "alpha_zre",
    "kb_zre",
    "b0_zre"
  ],
  "ell": [
    2033.496339593671,
    4093.4152871588512,
    6153.334234724031,
    8213.253182289212,
    10273.172129854393
  ],
  "y_source": "dl_ksz",
  "y_transform": "ln",
  "eps": 1e-30
}


## Record Build Provenance

In [9]:
def _plain(value):
    """Dataclass configs and stats -> JSON-safe dicts."""
    return asdict(value) if is_dataclass(value) else value


if REBUILD:
    build_info = {
        "configs": {
            "condense_config": _plain(cdcfg),
            "cl_config": _plain(clcfg),
            "build_config": _plain(xycfg),
        },
        "stats": {
            "condense": _plain(condense_stats),
            "cl": _plain(cl_stats),
            "build_xy": _plain(xy_stats),
        },
        "dataset": dataset_summary(H5_PATH),

        "environment": {
            "python": sys.version.split()[0],
            "numpy": np.__version__,
            "h5py": h5py.version.version,
            "seed": SEED,
        },
    }

    with BUILD_INFO_PATH.open("w", encoding="utf-8") as fh:
        json.dump(build_info, fh, indent=4, sort_keys=True, default=str)
        fh.write("\n")

    print(f"Build provenance written to:\n{BUILD_INFO_PATH}")

elif BUILD_INFO_PATH.exists():
    with BUILD_INFO_PATH.open("r", encoding="utf-8") as fh:
        build_info = json.load(fh)
    print(f"Loaded existing build provenance from:\n{BUILD_INFO_PATH}\n")
    print(json.dumps(build_info, indent=2)[:2000])

else:
    build_info = None
    print(f"No build provenance at:\n{BUILD_INFO_PATH}.\n")
    print("The dataset on disk predates this notebook.")

Build provenance written to:
/Users/robertxpearce/Desktop/Research/LEADS Lab/reionemu/reionemu-pasa-2026/results/pearce_2026_reionemu_mc_dropout/n_mc_201_epochs_1000_early_stop_150/records/dataset_build.json
